In [17]:
import pandas as pd
import numpy as np
import heapq

In [2]:
df = pd.read_excel('dados/base_analise.xlsx')

In [10]:
# Municípios sem PESA local
df_biv = df[df["ACESSO_LOCAL"] == 0].copy()

print("Municípios sem PESA local:", len(df_biv))

Municípios sem PESA local: 425


In [13]:
# Distribuição da carga de MG
print("CARGA DE MG")
print(df_biv["MG"].describe())

print("\nTEMPO ATÉ O PESA")
print(df_biv["TEMPO"].describe())

CARGA DE MG
count    425.000000
mean       5.774118
std        6.662397
min        0.000000
25%        1.000000
50%        4.000000
75%        8.000000
max       48.000000
Name: MG, dtype: float64

TEMPO ATÉ O PESA
count    425.000000
mean      27.376395
std       11.996764
min        6.800000
25%       19.000000
50%       25.200000
75%       34.300000
max       70.800000
Name: TEMPO, dtype: float64


In [14]:
print("\nDistribuição de MG:")
print(
    df_biv["MG"]
    .value_counts()
    .sort_index()
)


Distribuição de MG:
MG
0     66
1     46
2     43
3     45
4     39
5     28
6     19
7     25
8     18
9     19
10    13
11    10
12     6
13     9
14     4
15     2
16     3
17    11
18     1
19     2
21     1
22     1
23     1
24     2
25     1
27     1
28     1
29     1
31     1
32     1
33     1
35     1
37     1
42     1
48     1
Name: count, dtype: int64


In [16]:
print("\nPossíveis pontos de corte:")
print(
    df_biv[["MG", "TEMPO"]]
    .quantile([1/3, 2/3])
)


Possíveis pontos de corte:
           MG      TEMPO
0.333333  2.0  20.800000
0.666667  6.0  30.366667


In [18]:
# Carga epidemiológica
df_biv["CARGA_MG"] = pd.cut(
    df_biv["MG"],
    bins=[-np.inf, 2, 6, np.inf],
    labels=["Baixa", "Intermediária", "Alta"]
)

# Acessibilidade segundo tempo
df_biv["CAT_TEMPO"] = pd.cut(
    df_biv["TEMPO"],
    bins=[-np.inf, 20.8, 30.366667, np.inf],
    labels=["Menor", "Intermediário", "Maior"]
)

# Conferir o número de municípios em cada categoria
print("Carga de MG:")
print(df_biv["CARGA_MG"].value_counts(sort=False))

print("\nTempo de deslocamento:")
print(df_biv["CAT_TEMPO"].value_counts(sort=False))

Carga de MG:
CARGA_MG
Baixa            155
Intermediária    131
Alta             139
Name: count, dtype: int64

Tempo de deslocamento:
CAT_TEMPO
Menor            143
Intermediário    140
Maior            142
Name: count, dtype: int64


In [19]:
# Matriz
matriz = pd.crosstab(
    df_biv["CARGA_MG"],
    df_biv["CAT_TEMPO"],
    margins=True
)

print(matriz)

CAT_TEMPO      Menor  Intermediário  Maior  All
CARGA_MG                                       
Baixa             44             51     60  155
Intermediária     50             42     39  131
Alta              49             47     43  139
All              143            140    142  425


In [20]:
# Frequências observadas × esperadas
from scipy.stats import chi2_contingency

# Matriz sem os totais ("All")
matriz_obs = pd.crosstab(
    df_biv["CARGA_MG"],
    df_biv["CAT_TEMPO"]
)

# Qui-quadrado e frequências esperadas
chi2, p, gl, esperados = chi2_contingency(matriz_obs)

# Transformar os esperados em DataFrame
matriz_esp = pd.DataFrame(
    esperados,
    index=matriz_obs.index,
    columns=matriz_obs.columns
)

print("Frequências observadas:")
print(matriz_obs)

print("\nFrequências esperadas:")
print(matriz_esp.round(1))

Frequências observadas:
CAT_TEMPO      Menor  Intermediário  Maior
CARGA_MG                                  
Baixa             44             51     60
Intermediária     50             42     39
Alta              49             47     43

Frequências esperadas:
CAT_TEMPO      Menor  Intermediário  Maior
CARGA_MG                                  
Baixa           52.2           51.1   51.8
Intermediária   44.1           43.2   43.8
Alta            46.8           45.8   46.4


In [21]:
# Qui-quadrado e V de Cramér
import numpy as np
from scipy.stats import chi2_contingency

# Teste do qui-quadrado
chi2, p, gl, esperados = chi2_contingency(matriz_obs)

# V de Cramér
n = matriz_obs.to_numpy().sum()
r, c = matriz_obs.shape

v_cramer = np.sqrt(
    chi2 / (n * min(r - 1, c - 1))
)

print(f"Qui-quadrado: {chi2:.3f}")
print(f"Graus de liberdade: {gl}")
print(f"p-valor: {p:.4f}")
print(f"V de Cramér: {v_cramer:.3f}")

Qui-quadrado: 4.317
Graus de liberdade: 4
p-valor: 0.3649
V de Cramér: 0.071


In [22]:
# Quais células se afastam mais do esperado (resíduos padronizados ajustados)
# Proporções marginais
prop_linhas = matriz_obs.sum(axis=1).values / n
prop_colunas = matriz_obs.sum(axis=0).values / n

# Resíduos padronizados ajustados
residuos_ajustados = (
    matriz_obs.values - esperados
) / np.sqrt(
    esperados *
    (1 - prop_linhas[:, None]) *
    (1 - prop_colunas[None, :])
)

residuos_df = pd.DataFrame(
    residuos_ajustados,
    index=matriz_obs.index,
    columns=matriz_obs.columns
)

print(residuos_df.round(2))

CAT_TEMPO      Menor  Intermediário  Maior
CARGA_MG                                  
Baixa          -1.74          -0.01   1.75
Intermediária   1.32          -0.26  -1.06
Alta            0.49           0.27  -0.75


In [23]:
print("Até 50 min:", (df_biv["TEMPO"] <= 50).sum())
print("Acima de 50 min:", (df_biv["TEMPO"] > 50).sum())

print("\nPercentuais:")
print((df_biv["TEMPO"] <= 50).value_counts(normalize=True) * 100)

Até 50 min: 404
Acima de 50 min: 21

Percentuais:
TEMPO
True     95.058824
False     4.941176
Name: proportion, dtype: float64


In [24]:
import numpy as np
import pandas as pd

df_biv["TEMPO_50"] = pd.cut(
    df_biv["TEMPO"],
    bins=[-np.inf, 50, np.inf],
    labels=["≤50 min", ">50 min"]
)

matriz_50 = pd.crosstab(
    df_biv["CARGA_MG"],
    df_biv["TEMPO_50"],
    margins=True
)

print(matriz_50)

TEMPO_50       ≤50 min  >50 min  All
CARGA_MG                            
Baixa              142       13  155
Intermediária      128        3  131
Alta               134        5  139
All                404       21  425


In [25]:
from scipy.stats import chi2_contingency
import numpy as np

# Matriz sem totais
matriz_50 = pd.crosstab(
    df_biv["CARGA_MG"],
    df_biv["TEMPO_50"]
)

chi2, p, gl, esperados = chi2_contingency(matriz_50)

# V de Cramér
n = matriz_50.to_numpy().sum()
r, c = matriz_50.shape

v_cramer = np.sqrt(
    chi2 / (n * min(r - 1, c - 1))
)

print(f"Qui-quadrado: {chi2:.3f}")
print(f"Graus de liberdade: {gl}")
print(f"p-valor: {p:.4f}")
print(f"V de Cramér: {v_cramer:.3f}")

print("\nFrequências esperadas:")
print(
    pd.DataFrame(
        esperados,
        index=matriz_50.index,
        columns=matriz_50.columns
    ).round(2)
)

Qui-quadrado: 6.413
Graus de liberdade: 2
p-valor: 0.0405
V de Cramér: 0.123

Frequências esperadas:
TEMPO_50       ≤50 min  >50 min
CARGA_MG                       
Baixa           147.34     7.66
Intermediária   124.53     6.47
Alta            132.13     6.87


In [28]:
df.columns

Index(['Unnamed: 0', 'ACESSO_LOCAL', 'MULTIPLO_PESA', 'REGIAO', 'PESA', 'MUNI',
       'OBSERVACOES', 'LAT_MUNI', 'LON_MUNI', 'LAT_PESA', 'LON_PESA',
       'DISTANCIA', 'TEMPO', 'IBGE', 'POP10', 'POP11A59', 'POP60', 'POP_GERAL',
       'TOTAL_CASOS', 'TOTAL_10A', 'TOTAL_11A59', 'TOTAL_60', 'LEVE', 'MG',
       'MG_ORIGINAL', 'LEVE_10', 'MG_10', 'LEVE_11A59', 'MG_11A59', 'LEVE_60',
       'MG_60', 'TOTAL_AMPOLAS', 'N_PESA', 'CAT_TEMPO', 'PROP_POP_0A10',
       'PROP_POP_60MAIS', 'LOG_POP', 'PROP_MG', 'INCID_MEDIA_GERAL',
       'INCID_0A10', 'INCID_60MAIS', 'PROP_CASOS_0A10', 'PROP_CASOS_60MAIS',
       'TAXA_MG_100MIL', 'CLASS_IG', 'SORO_IG', 'EVOL_IG', 'PROP_CLASS_IGN',
       'PROP_SORO_IGN', 'PROP_EVOL_IGN', 'BAIXO_N'],
      dtype='str')

In [29]:
prioritarios = df_biv[
    (df_biv["CARGA_MG"] == "Alta") &
    (df_biv["TEMPO"] > 50)
].copy()

prioritarios[
    ["MUNI", "MG", "TOTAL_CASOS", "TEMPO", "POP_GERAL", "REGIAO"]
].sort_values("TEMPO", ascending=False)

,MUNI,MG,TOTAL_CASOS,TEMPO,POP_GERAL,REGIAO
328,IRAPUA,9,235,60.700000,7027,SAO JOSE DO RIO PRETO
344,PAULO DE FARIA,11,266,57.900000,7616,SAO JOSE DO RIO PRETO
67,JABORANDI,7,88,57.600000,6369,BARRETOS
421,GUARANTA,8,105,55.758065,6556,MARILIA
103,ANHEMBI,11,339,54.900000,5795,BOTUCATU


## 1. Caracterização epidemiológica do escorpionismo
* Total de acidentes escorpiônicos no período de 2019–2024.
* Distribuição anual dos casos.
* Incidência média anual por 100.000 habitantes.
* Distribuição dos casos segundo faixas etárias de interesse.
* Número e proporção de casos MG segundo a definição operacional.
* Distribuição dos principais indicadores:
    * por município;
    * por região.
* Identificação da heterogeneidade territorial da carga do escorpionismo.
* Produto principal: Tabela de caracterização epidemiológica, complementada por mapa da distribuição territorial do agravo.


(220, 51)

## 2. Caracterização da rede PESA
* Número de PESA identificados.
* Número e proporção de municípios:
    * com PESA local;
    * sem PESA local.
* Distribuição dos PESA segundo regiões.
* População residente em municípios com e sem PESA local.
* Número/proporção de acidentes ocorridos em municípios com e sem PESA local.
* Número/proporção de MG em municípios com e sem PESA local.
* Produto principal: Mapa da distribuição territorial da rede PESA.


425

# 3. Acessibilidade geográfica aos PESA
* População de análise: municípios sem PESA local.
* Número de municípios dependentes de deslocamento intermunicipal.
* Tempo estimado até o PESA de referência:
    * mediana;
    * intervalo interquartil;
    * mínimo e máximo.
* Distribuição dos municípios segundo estratos de tempo.
* Distribuição regional dos tempos de deslocamento.
* População residente em cada estrato de acessibilidade.
* Número de acidentes e MG em cada estrato de tempo.
* Identificação dos municípios/regiões com maiores tempos estimados.
* Produto principal: Mapa do tempo estimado de deslocamento até o PESA de referência.


## 4. Sobreposição entre carga epidemiológica e acessibilidade
* População de análise: municípios sem PESA local.
* Construção de duas variáveis categóricas:
    * carga acumulada de MG entre 2019–2024: menor, intermediária e maior;
    * tempo estimado até o PESA: menor, intermediário e maior.
* Construção da matriz bivariada 3 × 3, resultando em nove perfis territoriais.
* Para cada perfil:
    * número de municípios;
    * população;
    * total de acidentes;
    * número de MG;
    * participação na carga de MG.
* Identificação dos municípios classificados simultaneamente como maior carga de MG + maior tempo de deslocamento.
* Avaliação estatística da sobreposição:
* qui-quadrado de Pearson;
* V de Cramér;
* frequências observadas e esperadas;
* resíduos padronizados.
* Interpretação como sobreposição territorial entre carga e acessibilidade, e não como efeito do tempo sobre a gravidade.
* Produto principal: Mapa bivariado carga de MG × tempo de acesso, acompanhado da matriz 3 × 3.
